# 04 · Letting the model read the link's characters (character n-grams)

**In one sentence:** instead of us deciding which 40 numbers describe a link (notebook 03), we let the model look at **every short chunk of characters** in the link and learn by itself which chunks mean phishing.

### What is a character n-gram?
A run of *n* characters in a row. The link `paypa1.top` contains these 3-grams:
`pay`, `ayp`, `ypa`, `pa1`, `a1.`, `1.t`, `.to`, `top`.

We use every chunk of length 1 to 4. A link becomes a very long list of numbers, one per possible chunk, and almost all of them are zero.

### How the model works, in two pieces
1. **TF-IDF** gives each chunk in a link a score. A chunk scores high if it appears in *this* link but is **rare across all links**: rare chunks are more informative than common ones like `.com`.
2. **Logistic Regression** then learns **one weight per chunk**. A positive weight pushes towards phishing, a negative weight towards legitimate.

This is the simple, classical version of the character-level neural network (CNN) we build in Milestone 4.

**Published comparison:** a 2026 paper reports about 99.7% for URL-only models on this dataset (different split, so only a rough comparison).

### What happens below
1. Train and test over the same 5 splits as notebooks 02 and 03.
2. Look inside: which chunks did it learn?
3. Test properly whether it's better than notebook 03's XGBoost.
4. Combine the two models.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier

from phishguard.data import load_clean, TARGET
from phishguard.features import extract_features
from phishguard.ensemble import LogitAverage
from phishguard.experiment import run, summary_table, load_predictions, load_record
from phishguard.modeling import param_grid
from phishguard.stats import mcnemar

df = load_clean()
y = df[TARGET]
urls = df["URL"]

## Step 1 · Train and test over 5 splits

Two settings are tuned on validation, as always:
- `max_n`: the longest chunk used (3 or 4 characters);
- `C`: how simple the weights must stay (smaller = simpler). We try up to 30, so the best value isn't stuck at the edge of what we tried.

**How to read the table:** one row per split.
- `params`: the settings validation chose.
- `threshold`: the probability above which a link is called phishing, also chosen on validation.
- `accuracy`, `fp` (legitimate links wrongly flagged), `fn` (phishing missed).

In [2]:
def make_ngram(max_n, C):
    return make_pipeline(
        TfidfVectorizer(analyzer="char", ngram_range=(1, max_n), min_df=2,
                        sublinear_tf=True, dtype=np.float32),
        LogisticRegression(C=C, solver="liblinear", max_iter=3000),
    )

ngram = run("Char n-gram LogReg (raw URL)", make_ngram, param_grid(max_n=[3, 4], C=[10.0, 30.0, 100.0]),
            urls, y, df, features="raw URL", group="m2")
ngram.per_seed[["seed", "params", "threshold", "accuracy", "fp", "fn"]]

,seed,params,threshold,accuracy,fp,fn
0,42,"{'max_n': 4, 'C': 30.0}",0.30,0.997467,7,82
1,0,"{'max_n': 3, 'C': 10.0}",0.25,0.996670,14,103
2,1,"{'max_n': 3, 'C': 30.0}",0.36,0.996195,12,121
3,2,"{'max_n': 4, 'C': 30.0}",0.27,0.997085,5,97
4,3,"{'max_n': 4, 'C': 30.0}",0.20,0.996897,10,99


Around **99.7% on every split**, with roughly 5–14 false alarms and 80–120 missed phishing links out of ~35,000.

## Step 2 · What did it learn? (main split, seed 42)
Logistic Regression has one weight per chunk, so we can simply **sort the weights** and read the strongest ones. Large positive = pushes towards phishing; large negative = pushes towards legitimate.

In [3]:
vectorizer, logreg = ngram.reference.model.named_steps.values()
print(f"{len(vectorizer.vocabulary_):,} distinct n-grams")
weights = pd.Series(logreg.coef_[0], index=vectorizer.get_feature_names_out())
pd.DataFrame({
    "→ phishing": weights.nlargest(20).index, "weight": weights.nlargest(20).round(1).values,
    "→ legitimate": weights.nsmallest(20).index, "weight ": weights.nsmallest(20).round(1).values,
})

306,717 distinct n-grams


,→ phishing,weight,→ legitimate,weight
0,p://,30.7,ww,-29.3
1,p:,30.7,w,-26.1
2,p:/,30.7,www.,-19.2
3,tp:,30.7,ww.,-18.6
4,tp:/,30.7,/www,-17.8
5,ttp:,30.7,//ww,-17.8
6,m/,16.0,www,-17.7
7,com/,15.0,/ww,-17.5
8,om/,15.0,w.,-17.0
9,/a,10.2,://w,-15.8


**Look at the top of both lists:**
- The strongest phishing chunks are `p://`, `tp:`, `ttp:`: that's **`http://`**, the start of a link **without** the `s` of `https`.
- The strongest legitimate chunks are `www`, `ww.`, `//ww`: the **`www.`** part.

So the model mostly learned **how the link starts**, not what the website's name says. Real clues like `.top` (a cheap domain ending popular with attackers) only appear further down.

That's a warning sign: in this dataset every legitimate link starts with `https://www.`, but a real attacker can use `https://www.` too. We investigate this properly in Milestone 3.

## Step 3 · Is it really better than XGBoost on our 40 features?
The two models score almost the same, so we need a proper test before calling one better.

**McNemar's test** compares two models on the **same** test links, looking only at the links where they **disagree**:
- `only_a_right`: links the n-gram model got right and XGBoost got wrong;
- `only_b_right`: the reverse.

If the models were equally good, these two counts would be about equal. The **p-value** is the chance of seeing a gap this big if they really were equal: below 0.05 means the difference is probably real.

In [4]:
xgb_pred = load_predictions("m2", "XGBoost (URL features)")
ngram_pred = load_predictions("m2", "Char n-gram LogReg (raw URL)")
assert (xgb_pred["test_idx"] == ngram_pred["test_idx"]).all()
truth = y.loc[ngram_pred["test_idx"]].to_numpy()
correct_ngram = (ngram_pred["test_prob"] >= ngram_pred["threshold"]) == truth
correct_xgb = (xgb_pred["test_prob"] >= xgb_pred["threshold"]) == truth
mcnemar(correct_ngram, correct_xgb)

{'only_a_right': 38, 'only_b_right': 16, 'p_value': 0.003838265880326096}

**38 vs 16, p = 0.004:** on the main split, the n-gram model is genuinely a little better than XGBoost on features. But it's 38 vs 16 links out of ~35,000, so the gap is tiny in practice.

## Step 4 · Combining the two models
The two models read different things: XGBoost sees counts and structure, the n-gram model sees the characters. So we try combining them.

**How:** each model outputs a probability; we turn each into **log-odds** (`log(p / (1 − p))`, a scale where evidence adds up) and **average** them. This is fairer than averaging probabilities, because one model might be more confident in general than the other. `LogitAverage` in `src/phishguard/ensemble.py` does this.

It's a first, simple preview of the **hybrid model** in Milestone 5, which combines them properly by training them together.

In [5]:
xgb_settings = load_record("m2", "XGBoost (URL features)")["reference"]["params"]
ngram_settings = ngram.per_seed.set_index("seed").loc[42, "params"]
print("XGBoost:", xgb_settings, "| n-gram:", ngram_settings)

X_both = extract_features(urls).assign(URL=urls)
feature_cols = [c for c in X_both.columns if c != "URL"]
make_average = lambda: LogitAverage({
    "ngram": (lambda: make_ngram(**ngram_settings), "URL"),
    "xgb": (lambda: XGBClassifier(n_estimators=500, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                                  n_jobs=-1, random_state=42, **xgb_settings), feature_cols),
})
average = run("Average: n-gram + XGBoost (URL only)", make_average, [{}],
              X_both, y, df, features="raw URL + URL features", group="m2")
summary_table([ngram, average])

XGBoost: {'max_depth': 6, 'min_child_weight': 5} | n-gram: {'max_n': 4, 'C': 30.0}


,accuracy (%),f1 (%),roc_auc,pr_auc,mcc,tpr_at_fpr_1pct (%),"accuracy, seed 42 [95% CI]"
model,,,,,,,
Char n-gram LogReg (raw URL),99.69 ± 0.05,99.63 ± 0.06,0.9986 ± 0.0003,0.9988 ± 0.0002,0.9936 ± 0.0010,99.51 ± 0.08,"99.75% [99.69%, 99.80%]"
Average: n-gram + XGBoost (URL only),99.70 ± 0.03,99.64 ± 0.04,0.9986 ± 0.0002,0.9988 ± 0.0002,0.9938 ± 0.0006,99.51 ± 0.05,"99.75% [99.68%, 99.80%]"


## What we learned
1. **Character n-grams reach 99.69%** from the link text alone, about the same as the best published URL-only results.
2. **Combining with XGBoost gives 99.70%:** no meaningful gain. The differences are within the ± between splits.
3. **Reading the weights showed what it really learned:** mostly `http://` vs `https://www.`, the way links *start* in this dataset. That's the question Milestone 3 answers.
4. **Before calling a model better, test it:** notebook 05 does this for all the models together.